# 03 — Content Embeddings

**CineMind Phase 1 · Step 3** — Turn each movie's text into a content vector
using `intfloat/e5-small-v2`, then verify with a "movies like X" demo.

### Why content embeddings?

A content vector describes what a movie is *about* (title + genres).
Similar movies get similar vectors, so we can find "movies like Toy Story" even
for a brand-new movie with zero ratings — this is the solution to **item cold-start**.

> **Note:** First run downloads ~130 MB model (internet required, once only).

**Requires:** `data/items.csv` (from notebook 01)

**Outputs:** `artifacts/content_vecs.npy` · `artifacts/content_movie_ids.npy`

In [1]:
import os, sys

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Make src/ importable
if not any('src' in p for p in sys.path):
    sys.path.insert(0, 'src')

import cinemind_utils as cu
print("cwd:", os.getcwd())

import numpy as np
import pandas as pd
cu.ensure_dirs()

cwd: c:\Users\shaur\Desktop\My codes shaurya\Content Recommendation system\cinemind _phase1\cinemind_phase1


## 1 · Build text descriptions for each movie

In [2]:
items = pd.read_csv("data/items.csv")

def build_text(row):
    genres = row["genres"]
    if isinstance(genres, str):
        genres = genres.strip("[]").replace("'", "")
    # "passage:" prefix is what E5 models expect for documents
    return f"passage: {row['title']}. Genres: {genres}."

texts     = items.apply(build_text, axis=1).tolist()
movie_ids = items.movie_id.to_numpy()

print(f"Built {len(texts)} text descriptions.")
print("Example:", texts[0])

Built 1682 text descriptions.
Example: passage: Toy Story (1995). Genres: Animation, "Childrens", Comedy.


## 2 · Encode with E5

We normalise to unit length so dot product = cosine similarity.
This lets us rank movies purely by direction in embedding space, not magnitude.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/e5-small-v2")
vecs  = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
vecs = np.asarray(vecs, dtype=np.float32)
print(f"Done. Each movie -> {vecs.shape[1]}-dim content vector.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Done. Each movie -> 384-dim content vector.


## 3 · Save

In [4]:
np.save(os.path.join(cu.ARTIFACT_DIR, "content_vecs.npy"),      vecs)
np.save(os.path.join(cu.ARTIFACT_DIR, "content_movie_ids.npy"), movie_ids)
print(f"Saved content_vecs.npy  shape={vecs.shape}")

Saved content_vecs.npy  shape=(1682, 384)


## 4 · Demo — semantic similarity search (no ratings needed)

In [5]:
title_of  = dict(zip(items.movie_id, items.title))
id_to_row = {mid: i for i, mid in enumerate(movie_ids)}

def similar(movie_id, k=5):
    qi   = id_to_row[movie_id]
    sims = vecs @ vecs[qi]            # cosine similarity to every other movie
    top  = np.argsort(-sims)
    return [movie_ids[j] for j in top if movie_ids[j] != movie_id][:k]

demo_id = 1 if 1 in id_to_row else int(movie_ids[0])
print(f"Movies most similar to '{title_of[demo_id]}' (by title + genre text):")
for mid in similar(demo_id):
    print(f"  {title_of[mid]}")

print("\nMatched purely by text — no ratings needed.")
print("Next -> 04_two_tower.ipynb")

Movies most similar to 'Toy Story (1995)' (by title + genre text):
  Space Jam (1996)
  Goofy Movie, A (1995)
  Aladdin (1992)
  Flintstones, The (1994)
  Babe (1995)

Matched purely by text — no ratings needed.
Next -> 04_two_tower.ipynb
